# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [38]:
# Add code here 🔧
# load dataset
df = pd.read_csv('https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/airbnb_listings.csv')

# preview
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


In [39]:
df.shape

(459, 77)

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 77 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            459 non-null    int64  
 1   listing_url                                   459 non-null    object 
 2   scrape_id                                     459 non-null    int64  
 3   last_scraped                                  459 non-null    object 
 4   source                                        459 non-null    object 
 5   name                                          459 non-null    object 
 6   description                                   449 non-null    object 
 7   neighborhood_overview                         196 non-null    object 
 8   host_url                                      459 non-null    object 
 9   host_name                                     459 non-null    obj

### ✍️ Your Response: 🔧
1. The dataset includes Airbnb listing info such as location, property type, price, host details, and more.

2. Currently the dataset has 459 rows with 77 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [41]:
# Add code here 🔧
# drop unnecessary columns if they exist
df = df.drop(columns=[
    'id', 'listing_url', 'scrape_id', 'last_scraped', 'source',
    'name', 'description', 'neighborhood_overview',
    'host_url', 'host_name', 'host_about',
    'host_thumbnail_url', 'host_picture_url',
    'calendar_updated', 'calendar_last_scraped',
    'first_review', 'last_review', 'license'
], errors='ignore')

df = df.select_dtypes(include=['int64', 'float64'])

df.head()

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1,5,42.65789,-73.75370,4,1.0,2.0,2.0,70.0,28,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,104.0,2,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,2,2,42.64615,-73.75966,2,1.0,0.0,1.0,75.0,2,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,101.0,2,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,1,1,42.65559,-73.76506,4,1.0,1.0,2.0,110.0,1,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


In [42]:
df.shape

(459, 41)

### ✍️ Your Response: 🔧
1. I dropped columns such as IDs, URLs, and text fields that had things like name and host info. This is because they were not useful for predicting price and cannot be used ina regression model.

2. If they were included it could reduce the models accuracy because they do not have meaningful relationships with price.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [43]:
# Add code here 🔧
# correlation matrix
corr = df.corr()

# correlation with price (sorted)
corr['price'].sort_values(ascending=False)

,price
price,1.000000
accommodates,0.579588
beds,0.547032
bedrooms,0.499286
bathrooms,0.468030
estimated_revenue_l365d,0.249488
maximum_maximum_nights,0.122872
minimum_maximum_nights,0.112166
maximum_nights_avg_ntm,0.111271
availability_30,0.108409


### ✍️ Your Response: 🔧
1. Variables like accommodates, beds, bedrooms, and bathrooms had the strongest positive correlation with price. Whereas review_scores_communication, longitude, and review_scores_checkin and the most negative correlation.

2. The most useful predictors will be those that have the strongest postive correlation with price. For example, accommodates will be a great one!

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [51]:
# Add code here 🔧

# target variable
y = df['price']

# features (everything except price)
X = df.drop(columns=['price'])

#removing any missing values
X = X.dropna()
y = y.loc[X.index]

In [52]:
X.head()

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1,5,42.65789,-73.75370,4,1.0,2.0,2.0,28,1125,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,2,1125,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,2,2,42.64615,-73.75966,2,1.0,0.0,1.0,2,45,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,2,1125,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,1,1,42.65559,-73.76506,4,1.0,1.0,2.0,1,1125,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


In [53]:
y.head()

,price
0,70.0
1,104.0
2,75.0
3,101.0
4,110.0


### ✍️ Your Response: 🔧
1. I am using all variables that are numeric and do not have missing values except price as features. Especially important ones like beds, accommodates, and bedrooms.

2. This is a regression problem and not classification becasue it is predicting a continuous numeric value and not a categoyr.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [54]:
# Add code here 🔧
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [55]:
X_train.shape, X_test.shape

((292, 40), (74, 40))

## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [56]:
# Add code here 🔧
# create model
model = LinearRegression()

# train model
model.fit(X_train, y_train)

# make predictions
y_pred = model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [58]:
# Add code here 🔧
# calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)

MSE: 14803.245037774737
R²: 0.2340652754983975


### ✍️ Your Response: 🔧
1. My R squared score is 0.23 so that means my model explains about 23% of the variation in my price which is kind of low.

2. My MSE is 14803 which is pretty large and means that the model predictions can be quite far from the actaul prices. To try improving this I could try only using the most important feaures or just removing less relevant variables.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [62]:
# Add code here 🔧
# create table of features and coefficients
coef_df = pd.DataFrame({'Feature': X.columns,'Coefficient': model.coef_})

# sort by impact (largest first)
coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

coef_df.head(10)

,Feature,Coefficient
33,review_scores_value,38.765381
5,bathrooms,33.876175
28,review_scores_accuracy,20.275709
14,minimum_nights_avg_ntm,17.989741
4,accommodates,16.112832
7,beds,3.957925
10,minimum_minimum_nights,2.185758
34,instant_bookable,1.790310
16,availability_30,1.409497
37,calculated_host_listings_count_private_rooms,1.024570


In [63]:
coef_df.tail(10)

,Feature,Coefficient
39,reviews_per_month,-3.871906
6,bedrooms,-4.302017
32,review_scores_location,-6.589769
29,review_scores_cleanliness,-7.161289
27,review_scores_rating,-11.915734
30,review_scores_checkin,-14.299140
11,maximum_minimum_nights,-20.408640
31,review_scores_communication,-30.853858
3,longitude,-200.119452
2,latitude,-311.257265


### ✍️ Your Response: 🔧
1. The features that increased price the most was review_scores_value and bathrooms.

2. There were several that had very strong negative coefficients including latitude and longitude. But features like bedrooms having a negative relationship was very suprising as you would expect the opposite.

3. Listings with better value ratings, higher capacity, and have more bathrooms often have a higher price. It also seems that factors like neightborhood differences also have a huge impact.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [67]:
# select top features
X_refined = X[['latitude', 'longitude', 'review_scores_value', 'bathrooms', 'review_scores_communication']]

# split again
from sklearn.model_selection import train_test_split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_refined, y, test_size=0.2, random_state=42)

# train model
from sklearn.linear_model import LinearRegression
model_r = LinearRegression()
model_r.fit(X_train_r, y_train_r)

# predict
y_pred_r = model_r.predict(X_test_r)

# evaluate
from sklearn.metrics import mean_squared_error, r2_score
mse_r = mean_squared_error(y_test_r, y_pred_r)
r2_r = r2_score(y_test_r, y_pred_r)

print("Refined MSE:", mse_r)
print("Refined R²:", r2_r)

Refined MSE: 20339.29197938483
Refined R²: -0.05237533790966564


### ✍️ Your Response: 🔧
1. I kept that features that seemed to have the strongest absoulte impact on price which includes:latitude, longitude, review_scores_value, bathrooms, and review_scores_communication

2. No the model performance did not improve. The R squared went from 0.23 to -0.05. The MSE went from 14803 to 20339. This happened because we reduced to many features and took away info from the model.

3. I would recommend the orignal to stakeholders because it was more accurate.

4. This relates because I got to use EDA and regression techniquest to find which features impact Airbnb pricing. Overall finding patterns that influence business decisions and evaluating models.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. This model helped answer what factors impact the Airbnb listing price and how we can predict price based off features.

2. I would recommend they focus on increasing the value of their listings and doing what they can to improve on the feaures that have the strongest impact like review scores.

3. To improve the model I could add more relevnat features or try even more advanced models like decisions tress.

4. As mentioned, this relates to my learning outcomes because I used data analysis to find patterns and evaluated how well those patterns and factors impacted predicted outcomes.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"